### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [1]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


class ShipmentError(Exception):
    pass


class RowFormatError(ShipmentError):
    pass


class WeightError(ShipmentError):
    pass


class PriorityError(ShipmentError):
    pass


class RegionError(ShipmentError):
    pass


def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    try:
        parts= row.split(',')
        if len(parts) != 5:
            raise RowFormatError("Некорректный формат строки")
        shipment_id,client,weight_str,priority,region = parts

    # TODO: при ошибке конвертации weight использовать raise ... from ...
        try:
            weight = float(weight_str)
        except ValueError as e:
            raise WeightError(f"Некорректное значение веса: '{weight_str}'") from e
        if weight <= 0:
            raise WeightError("Вес должен быть положительным")
        valid_priorities = {'standard','express','vip'}
        if priority not in valid_priorities:
            raise PriorityError("Недопустимый приоритет")
        valid_regions ={'RU','KZ','BY'}
        if region not in valid_regions:
            raise RegionError("Недопустимый регион")
  
        return { 
            'shipment_id': shipment_id,
            'client': client,
            'weight': weight,
            'priority': priority,
            'region': region
        }
    except ShipmentError as e:
        raise e
    except Exception as e:
        raise RowFormatError ("Ошибка при обработке строки")
  
def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []
    for row in rows:
        try:
            shipment = parse_shipment(row)
            shipments.append(shipment)
        except ShipmentError as e:
            errors.append((row,type(e).__name__,str(e)))
        except Exception as e:
            errors.append((row, 'UnknownError', str(e)))
    return shipments,errors

# TODO: вызвать load_shipments(rows)
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]


shipments,errors = load_shipments(rows)

# TODO: вывести число валидных отгрузок и число ошибок
print(f"Количество корректных отгрузок: {len(shipments)}")
print(f"Число ошибок: {len(errors)}")

# TODO: вывести ошибки по типам
error_counts = {}
for _, error_type, _ in errors:
    error_counts[error_type] = error_counts.get(error_type, 0) + 1

print("\nОшибки по типам:")
for error_type, count in error_counts.items():
    print(f"  {error_type}: {count}")

# TODO: посчитать premium_weight только для express/vip
premium_weight = sum(
    shipment['weight']
    for shipment in shipments
    if shipment['priority'] in {'express','vip'}
)
print(f"\nСуммарный вес (express + vip): {premium_weight}")

# TODO: найти клиента-лидера по суммарному весу
client_weights = {}
for shipment in shipments:
    client = shipment ['client']
    weight = shipment['weight']
    client_weights[client] = client_weights.get(client,0) + weight
if client_weights:
    leader = max(client_weights,key = client_weights.get )
    leader_weight = client_weights[leader]
    print(f"Клиент-лидер по суммарному весу: {leader} ({leader_weight})")
else:
    print("Клиент-лидер не найден (нет корректных записей)")

Количество корректных отгрузок: 3
Число ошибок: 4

Ошибки по типам:
  WeightError: 2
  PriorityError: 1
  RegionError: 1

Суммарный вес (express + vip): 22.0
Клиент-лидер по суммарному весу: Acme (16.5)
